Part 1- Environment setup

In [ ]:
!nvidia-smi

In [ ]:
!pip install -q \
  transformers \
  datasets \
  accelerate \
  bitsandbytes \
  peft \
  trl \
  sentencepiece \
  scipy \
  scikit-learn

In [ ]:
!pip install textstat

In [ ]:
!pip install openai

In [ ]:
!pip install rouge-score bert-score -q

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
# model_name = "mistralai/Mistral-7B-Instruct-v0.2"

# bnb_config = BitsAndBytesConfig(
#     load_in_4bit=True,
#     bnb_4bit_quant_type="nf4",
#     bnb_4bit_compute_dtype=torch.float16,
#     bnb_4bit_use_double_quant=True
# )

# model = AutoModelForCausalLM.from_pretrained(
#     model_name,
#     quantization_config=bnb_config,
#     device_map="auto"
# )

In [ ]:
# tokenizer = AutoTokenizer.from_pretrained(model_name)
# tokenizer.padding_side = "left"
# tokenizer.pad_token = tokenizer.eos_token

In [ ]:
# messages = [
#     {
#         "role": "system",
#         "content": "You are a medical assistant that explains clinical text in simple language for patients."
#     },
#     {
#         "role": "user",
#         "content": "Simplify this clinical sentence: The patient presents with hyperglycemia secondary to insulin resistance."
#     }
# ]

# prompt = tokenizer.apply_chat_template(
#     messages,
#     tokenize=False,
#     add_generation_prompt=True
# )

# print(prompt)

In [ ]:
# inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

In [ ]:
# outputs = model.generate(
#     **inputs,
#     max_new_tokens=120,
#     temperature=0.7,
#     do_sample=True
# )

# result = tokenizer.decode(outputs[0], skip_special_tokens=True)

# print(result)

In [ ]:
# baseline_outputs = []

# baseline_outputs.append({
#     "clinical_text": "The patient presents with hyperglycemia secondary to insulin resistance.",
#     "baseline_model_output": result
# })

# baseline_outputs

Part 2- Dataset pipeline

In [ ]:
from datasets import load_dataset
ds = load_dataset("armanc/pubmed-rct20k")

print(ds)

In [ ]:
ds["train"][0]

In [ ]:
biomedical_sentences = []

for example in ds["train"]:
    sentence = example["text"]

    if len(sentence) > 80:
        biomedical_sentences.append(sentence)

    if len(biomedical_sentences) >= 3000:
        break

print("Collected:", len(biomedical_sentences))

In [ ]:
import re

def clean_text(text):

    text = text.replace("@", "")
    text = text.replace("-LSB-", "")
    text = text.replace("-RSB-", "")

    text = re.sub(r"\s+", " ", text)

    return text.strip()


cleaned_sentences = [clean_text(s) for s in biomedical_sentences]

In [ ]:
import random

for _ in range(5):
    print(random.choice(cleaned_sentences))

In [ ]:
paragraph_inputs = []

for i in range(0, 450, 3):
    paragraph = " ".join(cleaned_sentences[i:i+3])

    if len(paragraph) > 150:
        paragraph_inputs.append(paragraph)

print("Paragraphs:", len(paragraph_inputs))

In [ ]:
sentence_inputs = cleaned_sentences[:250]
print("Sentence samples:", len(sentence_inputs))

In [ ]:
inputs = sentence_inputs + paragraph_inputs
print("Total inputs:", len(inputs))

In [ ]:
# import torch
# from transformers import AutoTokenizer, AutoModelForCausalLM

# model_id = "Qwen/Qwen2.5-7B-Instruct"
# tokenizer = AutoTokenizer.from_pretrained(model_id)

# model = AutoModelForCausalLM.from_pretrained(
#     model_id,
#     torch_dtype=torch.float16,
#     device_map="auto"
# )
print("Skipping Qwen teacher model. Dataset will be uploaded manually later on.")

In [ ]:
import torch

def simplify_text(batch):

    prompts = [
        f"""Rewrite the biomedical text so that a patient can understand it.

Rules:
- Use simple language
- Do not add medical advice
- Do not add new facts
- Preserve disease and drug names

Biomedical text:
{text}

Patient explanation:"""
        for text in batch
    ]

    tokens = tokenizer(
        prompts,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=512
    ).to(model.device)

    with torch.no_grad():

        outputs = model.generate(
            **tokens,
            max_new_tokens=120,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    decoded = tokenizer.batch_decode(outputs, skip_special_tokens=True)

    results = []
    for d in decoded:
        results.append(d.split("Patient explanation:")[-1].strip())

    return results

In [ ]:
# outputs = []

# batch_size = 8

# for i in range(0, len(inputs), batch_size):

#     batch = inputs[i:i+batch_size]

#     batch_outputs = simplify_text(batch)

#     outputs.extend(batch_outputs)

#     print(f"Processed {min(i+batch_size, len(inputs))}/{len(inputs)}")

In [ ]:
# dataset = []

# for inp, out in zip(inputs, outputs):
#     dataset.append({
#         "instruction": "Simplify the clinical text so a patient can understand it.",
#         "input": inp,
#         "output": out
#     })
# print(len(dataset))

In [ ]:
# import json
# with open("medical_simplification_dataset.jsonl", "w") as f:
#     for row in dataset:
#         f.write(json.dumps(row) + "\n")

# print("Dataset saved")

In [ ]:
# from sklearn.model_selection import train_test_split

# train_data, val_data = train_test_split(
#     dataset,
#     test_size=0.1,
#     random_state=42
# )
# print(len(train_data), len(val_data))

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')

In [ ]:
# import json

# path = "/content/drive/MyDrive/medical_simplification_dataset.jsonl"

# with open(path, "w") as f:
#     for row in dataset:
#         f.write(json.dumps(row) + "\n")

# print("Saved to Google Drive")

In [ ]:
# from datasets import load_dataset

# dataset = load_dataset(
#     "json",
#     data_files="/content/drive/MyDrive/medical_simplification_dataset.jsonl"
# )

Part 3- SFT Training

In [ ]:
from google.colab import files
uploaded = files.upload()

In [ ]:
medical_simplification_dataset.jsonl

In [ ]:
from datasets import load_dataset

dataset = load_dataset(
    "json",
    data_files="medical_simplification_dataset.jsonl"
)

dataset

In [ ]:
dataset = dataset["train"].train_test_split(test_size=0.1, seed=42)

dataset

In [ ]:
dataset["validation"] = dataset.pop("test")

In [ ]:
dataset["train"][0]

In [ ]:
from transformers import AutoTokenizer

model_name = "mistralai/Mistral-7B-Instruct-v0.2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

In [ ]:
def format_chat(example):

    messages = [
        {"role": "system", "content": example["instruction"]},
        {"role": "user", "content": example["input"]},
        {"role": "assistant", "content": example["output"]}
    ]

    example["text"] = tokenizer.apply_chat_template(
        messages,
        tokenize=False
    )

    return example

In [ ]:
dataset = dataset.map(format_chat)

In [ ]:
print(dataset["train"][0]["text"])

In [ ]:
dataset["train"].column_names

In [ ]:
for i in range(3):
    print(dataset["train"][i]["text"])
    print("\n#########\n")

In [ ]:
# import torch
# from transformers import AutoModelForCausalLM, BitsAndBytesConfig
# from peft import prepare_model_for_kbit_training

In [ ]:
import gc, torch

if 'model' in globals():
    del model
gc.collect()
torch.cuda.empty_cache()

In [ ]:
# from transformers import AutoModelForCausalLM, BitsAndBytesConfig
# import torch

# bnb_config = BitsAndBytesConfig(
#     load_in_4bit=True,
#     bnb_4bit_quant_type="nf4",
#     bnb_4bit_compute_dtype=torch.float16,
#     bnb_4bit_use_double_quant=True
# )

# model = AutoModelForCausalLM.from_pretrained(
#     model_name,
#     quantization_config=bnb_config,
#     device_map="auto",
#     trust_remote_code=True,
#     low_cpu_mem_usage=True
# )

In [ ]:
# model.config.use_cache = False

In [ ]:
import torch, gc
gc.collect()
torch.cuda.empty_cache()

torch.backends.cuda.matmul.allow_tf32 = True

In [ ]:
from peft import prepare_model_for_kbit_training, LoraConfig, get_peft_model

# Step 1: prepare for kbit
# model = prepare_model_for_kbit_training(
#     model,
#     use_gradient_checkpointing=True,
#     gradient_checkpointing_kwargs={"use_reentrant": False}
# )

In [ ]:
print(model.device)

In [ ]:
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj"
    ],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

In [ ]:
# model = get_peft_model(model, lora_config)

In [ ]:
# model.print_trainable_parameters()

In [ ]:
# from trl import SFTConfig

# training_args = SFTConfig(
#     dataset_text_field="text",
#     max_length=512,
#     output_dir="./mistral-medical-simplifier",
#     per_device_train_batch_size=2,
#     per_device_eval_batch_size=2,
#     gradient_accumulation_steps=4,
#     num_train_epochs=3,
#     learning_rate=2e-4,
#     fp16=True,
#     bf16=False,
#     optim="paged_adamw_8bit",
#     logging_steps=10,
#     eval_strategy="steps",
#     eval_steps=50,
#     save_strategy="epoch",
#     warmup_steps=50,
#     lr_scheduler_type="cosine",
#     report_to="none"
# )

In [ ]:
# from trl import SFTTrainer

# trainer = SFTTrainer(
#     model=model,
#     processing_class=tokenizer,
#     train_dataset=dataset["train"],
#     eval_dataset=dataset["validation"],
#     args=training_args
# )

In [ ]:
  # trainer.train()

In [ ]:
# trainer.model.save_pretrained("mistral-medical-lora")
# tokenizer.save_pretrained("mistral-medical-lora")

In [ ]:
# from google.colab import files
# import shutil

# shutil.make_archive("mistral-medical-lora", 'zip', "mistral-medical-lora")
# files.download("mistral-medical-lora.zip")

Part 4- Evaluation

In [ ]:
from datasets import load_dataset

dataset = load_dataset(
    "json",
    data_files="medical_simplification_dataset.jsonl"
)
dataset = dataset["train"]
print(dataset[0])
print("Dataset size:", len(dataset))

In [ ]:
import pandas as pd

eval_df = pd.DataFrame({
    "clinical_text": [x["input"] for x in dataset],
    "reference_simplification": [x["output"] for x in dataset]
})
eval_df.head()

In [ ]:
from google.colab import files
uploaded = files.upload()

In [ ]:
from google.colab import files
uploaded = files.upload()

In [ ]:
!unzip mistral-medical-lora.zip

In [ ]:
!mkdir -p mistral-medical-lora && mv adapter_config.json adapter_model.safetensors tokenizer.json tokenizer_config.json chat_template.jinja README.md mistral-medical-lora/

In [ ]:
!ls mistral-medical-lora

In [ ]:
from datasets import load_dataset

dataset = load_dataset(
    "json",
    data_files="medical_simplification_dataset.jsonl"
)
dataset = dataset["train"]
print("Dataset size:", len(dataset))
print(dataset[0])

In [ ]:
import pandas as pd

eval_df = pd.DataFrame({
    "clinical_text": [x["input"] for x in dataset],
    "reference": [x["output"] for x in dataset]
})
eval_df.head()

In [ ]:
eval_df = eval_df.sample(100, random_state=42).reset_index(drop=True)
print("Evaluation samples:", len(eval_df))

In [ ]:
import gc, torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel

if 'model' in globals():
    del model
gc.collect()
torch.cuda.empty_cache()

model_name = "mistralai/Mistral-7B-Instruct-v0.2"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
    low_cpu_mem_usage=True
)

# Loading trained LoRA adapter on top
# model = PeftModel.from_pretrained(base_model, "mistral-medical-lora")
# model.eval()

In [ ]:
def create_prompt(text):
    prompt = f"""<s>[INST] Simplify the clinical text so a patient can understand it.

{text}

[/INST]"""
    return prompt

In [ ]:
text = eval_df["clinical_text"][0]

prompt = create_prompt(text)
inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

outputs = base_model.generate(
    **inputs,
    max_new_tokens=150,
    do_sample=True,
    temperature=0.3
)

response = tokenizer.decode(outputs[0], skip_special_tokens=True)
print(response)

In [ ]:
baseline_outputs = []

for text in eval_df["clinical_text"][:20]:
    prompt = create_prompt(text)
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

    outputs = base_model.generate(
        **inputs,
        max_new_tokens=150,
        do_sample=True,
        temperature=0.3
    )

    response = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[-1]:],
        skip_special_tokens=True
    )

    baseline_outputs.append(response)

In [ ]:
eval_df.loc[:19, "baseline_output"] = baseline_outputs

In [ ]:
from peft import PeftModel

# finetuned_model = PeftModel.from_pretrained(
#     base_model,
#     "mistral-medical-lora"
# )

# finetuned_model.eval()
finetuned_model = PeftModel.from_pretrained(base_model, "mistral-medical-lora")
finetuned_model.eval()

In [ ]:
text = eval_df["clinical_text"][0]

prompt = create_prompt(text)
inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

outputs = finetuned_model.generate(
    **inputs,
    max_new_tokens=150,
    do_sample=True,
    temperature=0.3
)
response = tokenizer.decode(outputs[0], skip_special_tokens=True)
print(response)

In [ ]:
finetuned_outputs = []

for text in eval_df["clinical_text"][:20]:

    prompt = create_prompt(text)
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

    outputs = finetuned_model.generate(
        **inputs,
        max_new_tokens=150,
        do_sample=True,
        temperature=0.3
    )

    response = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[-1]:],
        skip_special_tokens=True
    )

    finetuned_outputs.append(response)

In [ ]:
eval_df.loc[:19, "finetuned_output"] = finetuned_outputs

In [ ]:
import textstat

In [ ]:
def clean_output(text):
    return text.split("[/INST]")[-1].strip()

eval_df["baseline_output"] = eval_df["baseline_output"].apply(clean_output)
eval_df["finetuned_output"] = eval_df["finetuned_output"].apply(clean_output)

In [ ]:
import textstat

eval_df["clinical_grade"] = eval_df["clinical_text"][:20].apply(
    lambda x: textstat.flesch_kincaid_grade(str(x))
)

eval_df["baseline_grade"] = eval_df["baseline_output"][:20].apply(
    lambda x: textstat.flesch_kincaid_grade(str(x))
)

eval_df["finetuned_grade"] = eval_df["finetuned_output"][:20].apply(
    lambda x: textstat.flesch_kincaid_grade(str(x))
)

In [ ]:
eval_df[[
    "clinical_grade",
    "baseline_grade",
    "finetuned_grade"
]].head()

In [ ]:
print("Average Clinical Grade:", eval_df["clinical_grade"][:20].mean())
print("Average Baseline Grade:", eval_df["baseline_grade"][:20].mean())
print("Average Finetuned Grade:", eval_df["finetuned_grade"][:20].mean())

In [ ]:
eval_df.loc[:19, "baseline_improvement"] = (
    eval_df["clinical_grade"][:20] - eval_df["baseline_grade"][:20]
)
eval_df.loc[:19, "finetuned_improvement"] = (
    eval_df["clinical_grade"][:20] - eval_df["finetuned_grade"][:20]
)
print("Baseline Improvement:", eval_df["baseline_improvement"][:20].mean())
print("Finetuned Improvement:", eval_df["finetuned_improvement"][:20].mean())

In [ ]:
from rouge_score import rouge_scorer

scorer = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=True)

baseline_rouge1, baseline_rougeL = [], []
finetuned_rouge1, finetuned_rougeL = [], []

for i in range(20):
    ref = eval_df["reference"][i]
    base = eval_df["baseline_output"][i]
    fine = eval_df["finetuned_output"][i]

    base_scores = scorer.score(ref, base)
    fine_scores = scorer.score(ref, fine)

    baseline_rouge1.append(base_scores["rouge1"].fmeasure)
    baseline_rougeL.append(base_scores["rougeL"].fmeasure)
    finetuned_rouge1.append(fine_scores["rouge1"].fmeasure)
    finetuned_rougeL.append(fine_scores["rougeL"].fmeasure)

print("Baseline ROUGE-1:", round(sum(baseline_rouge1)/len(baseline_rouge1), 4))
print("Baseline ROUGE-L:", round(sum(baseline_rougeL)/len(baseline_rougeL), 4))
print("Finetuned ROUGE-1:", round(sum(finetuned_rouge1)/len(finetuned_rouge1), 4))
print("Finetuned ROUGE-L:", round(sum(finetuned_rougeL)/len(finetuned_rougeL), 4))

In [ ]:
from bert_score import score as bert_score

refs = eval_df["reference"][:20].tolist()
baseline_preds = eval_df["baseline_output"][:20].tolist()
finetuned_preds = eval_df["finetuned_output"][:20].tolist()

_, _, base_F1 = bert_score(baseline_preds, refs, lang="en", verbose=False)
_, _, fine_F1 = bert_score(finetuned_preds, refs, lang="en", verbose=False)

print("Baseline BERTScore F1:", round(base_F1.mean().item(), 4))
print("Finetuned BERTScore F1:", round(fine_F1.mean().item(), 4))

In [ ]:
# API keys
import os
from getpass import getpass

os.environ["OPENAI_API_KEY"] = getpass("Enter OpenAI API key:")

In [ ]:
from openai import OpenAI
client = OpenAI()

In [ ]:
def create_judge_prompt(clinical, simplified):

    return f"""
You are a medical expert evaluating a simplified explanation.

Clinical text:
{clinical}

Simplified explanation:
{simplified}

Score from 1 to 10:

Accuracy: medical meaning preserved
Simplicity: understandable to patient
Faithfulness: no hallucinated facts

Return ONLY numbers in this format:

Accuracy: X
Simplicity: X
Faithfulness: X
"""

In [ ]:
def judge_with_gpt(prompt):

    response = client.chat.completions.create(
        model="gpt-5-nano",
        messages=[
            {"role": "user", "content": prompt}
        ],
        temperature=1
    )

    return response.choices[0].message.content

In [ ]:
clinical = eval_df["clinical_text"][0]
simplified = eval_df["finetuned_output"][0]

prompt = create_judge_prompt(clinical, simplified)

response = judge_with_gpt(prompt)

print(response)

In [ ]:
import re

def extract_scores(text):

    acc = re.search(r"Accuracy:\s*(\d+)", text)
    sim = re.search(r"Simplicity:\s*(\d+)", text)
    faith = re.search(r"Faithfulness:\s*(\d+)", text)

    acc = int(acc.group(1)) if acc else None
    sim = int(sim.group(1)) if sim else None
    faith = int(faith.group(1)) if faith else None

    return acc, sim, faith

In [ ]:
import math

judge_accuracy = []
judge_simplicity = []
judge_faithfulness = []

for i in range(20):

    clinical = eval_df["clinical_text"][i]
    simplified = eval_df["finetuned_output"][i]

    prompt = create_judge_prompt(clinical, simplified)
    response = judge_with_gpt(prompt)

    acc, sim, faith = extract_scores(response)

    if None in (acc, sim, faith):
        print(f"[Warning] Row {i}: could not parse GPT scores - storing NaN")
        acc  = math.nan
        sim  = math.nan
        faith = math.nan

    judge_accuracy.append(acc)
    judge_simplicity.append(sim)
    judge_faithfulness.append(faith)

In [ ]:
eval_df.loc[:19, "judge_accuracy"] = judge_accuracy
eval_df.loc[:19, "judge_simplicity"] = judge_simplicity
eval_df.loc[:19, "judge_faithfulness"] = judge_faithfulness

In [ ]:
print("Average Accuracy:",
      eval_df["judge_accuracy"].mean())

print("Average Simplicity:",
      eval_df["judge_simplicity"].mean())

print("Average Faithfulness:",
      eval_df["judge_faithfulness"].mean())

In [ ]:
results = {
    "Clinical Grade Level": eval_df["clinical_grade"].mean(),
    "Baseline Grade Level": eval_df["baseline_grade"].mean(),
    "Finetuned Grade Level": eval_df["finetuned_grade"].mean(),
    "Baseline Readability Improvement": eval_df["baseline_improvement"].mean(),
    "Finetuned Readability Improvement": eval_df["finetuned_improvement"].mean(),
    "Baseline ROUGE-1": round(sum(baseline_rouge1)/len(baseline_rouge1), 4),
    "Finetuned ROUGE-1": round(sum(finetuned_rouge1)/len(finetuned_rouge1), 4),
    "Baseline ROUGE-L": round(sum(baseline_rougeL)/len(baseline_rougeL), 4),
    "Finetuned ROUGE-L": round(sum(finetuned_rougeL)/len(finetuned_rougeL), 4),
    "Baseline BERTScore F1": round(base_F1.mean().item(), 4),
    "Finetuned BERTScore F1": round(fine_F1.mean().item(), 4),
    "LLM Judge Accuracy": eval_df["judge_accuracy"].mean(),
    "LLM Judge Simplicity": eval_df["judge_simplicity"].mean(),
    "LLM Judge Faithfulness": eval_df["judge_faithfulness"].mean()
}

results

In [ ]:
results_df = pd.DataFrame(list(results.items()), columns=["Metric", "Score"])
results_df

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8,5))

plt.bar(
    ["Clinical", "Baseline", "Finetuned"],
    [
        results["Clinical Grade Level"],
        results["Baseline Grade Level"],
        results["Finetuned Grade Level"]
    ]
)

plt.title("Readability Grade Comparison")
plt.ylabel("Grade Level")
plt.show()

In [ ]:
eval_df.to_csv("evaluation_results.csv", index=False)

In [ ]:
results_df.to_csv("evaluation_summary.csv", index=False)

In [ ]:
from google.colab import files

# Saving full eval dataframe
eval_df[:20].to_csv("evaluation_results.csv", index=False)

# Saving summary scores
results_df = pd.DataFrame(list(results.items()), columns=["Metric", "Score"])
results_df.to_csv("evaluation_summary.csv", index=False)

# Downloading both files
files.download("evaluation_results.csv")
files.download("evaluation_summary.csv")

Part 5- Inference Optimization

In [ ]:
import torch, gc
gc.collect()
torch.cuda.empty_cache()

In [ ]:
from google.colab import files
uploaded = files.upload()

In [ ]:
!unzip mistral-medical-lora.zip

In [ ]:
!mkdir -p mistral-medical-lora && mv adapter_config.json adapter_model.safetensors tokenizer.json tokenizer_config.json mistral-medical-lora/

In [ ]:
!ls mistral-medical-lora

In [ ]:
# !pip install transformers==4.38.2 -qy

In [ ]:
# !pip install peft==0.18.1 -q

In [ ]:
# pip install -U bitsandbytes>=0.46.1

In [ ]:
# import json
# with open("/content/mistral-medical-lora/adapter_config.json") as f:
#     config = json.load(f)
# print(json.dumps(config, indent=2))

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel
import torch

base_model_id = "mistralai/Mistral-7B-Instruct-v0.2"
lora_path = "/content/mistral-medical-lora"

# Loading in 4-bit config to fit within T4's 15GB VRAM
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)

tokenizer = AutoTokenizer.from_pretrained(base_model_id)

base_model = AutoModelForCausalLM.from_pretrained(
    base_model_id,
    quantization_config=bnb_config,
    # Using single GPU, no CPU offload
    device_map="cuda:0"
)

model = PeftModel.from_pretrained(base_model, lora_path)
model.eval()

In [ ]:
# Prompt Template- same as training
def build_prompt(text):
    return f"""### Instruction:
Simplify the following clinical text into patient-friendly explanation.

### Input:
{text}

### Response:
"""

In [ ]:
# LoRA inference

def generate(text, model):
    prompt = build_prompt(text)
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

    with torch.no_grad():
        output = model.generate(
    **inputs,
    max_new_tokens=200,
    do_sample=True,
    temperature=0.3,
    top_p=0.9
)

    return tokenizer.decode(output[0], skip_special_tokens=True)

In [ ]:
# Merging LoRA with Base Model
merged_model = model.merge_and_unload()

In [ ]:
# Save Merged Model
save_path = "/content/merged_model"
merged_model.save_pretrained(save_path)
tokenizer.save_pretrained(save_path)

In [ ]:
# Reload for Fast Inference
fast_model = AutoModelForCausalLM.from_pretrained(
    save_path,
    torch_dtype=torch.float16,
    device_map="auto"
)
fast_model.eval()

In [ ]:
# Latency Comparison
import time
sample = "The patient presents with myocardial infarction."

# LoRA
start = time.time()
generate(sample, model)
lora_time = time.time() - start

# Merged
start = time.time()
generate(sample, fast_model)
merged_time = time.time() - start

print(f"LoRA time: {lora_time:.2f}s")
print(f"Merged time: {merged_time:.2f}s")

In [ ]:
# Guardrails
def safe_generate(text, model):
    output = generate(text, model)

    # Basic guardrails
    if "diagnosis" in output.lower():
        output = "This system does not provide medical diagnosis.\n\n" + output

    return {
        "input": text,
        "output": output.strip(),
        "note": "Educational purpose only. Not medical advice."
    }